<a href="https://colab.research.google.com/github/wykyty/learning-journey/blob/main/01-sparse-autoencoders/training_sae_t5_nnsight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training a Sparse Autoencoder on T5-large with nnsight

This notebook trains a Sparse Autoencoder (SAE) on the **encoder MLP activations** of T5-large, using [nnsight](https://nnsight.net/) for activation capture. Since SAELens only supports decoder-only models via HookedTransformer, we implement the SAE training from scratch in PyTorch while following SAELens conventions.

**Key differences from the SAELens tutorial:**
- Uses nnsight to wrap T5 (an encoder-decoder model) instead of TransformerLens
- Custom SAE class instead of SAELens's `StandardTrainingSAE`
- Custom activation collection pipeline using nnsight's trace API
- Manual training loop instead of SAELens's `SAETrainingRunner`

**Target:** Encoder block 12 MLP output (`model.encoder.block[12].layer[1]`), d_model=1024

## 1. Setup and Dependencies

In [ ]:
try:
    import google.colab  # type: ignore
    %pip install nnsight torch datasets safetensors tqdm matplotlib
except Exception:
    from IPython import get_ipython  # type: ignore
    ipython = get_ipython()
    assert ipython is not None
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass
from pathlib import Path
from tqdm.auto import tqdm
from safetensors.torch import save_file, load_file

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 2. Load T5-large via nnsight

nnsight wraps HuggingFace models using `LanguageModel`, which accepts `automodel` to specify the model class. For T5, we **must** use `AutoModelForSeq2SeqLM` — the default `AutoModelForCausalLM` will not work.

The `dispatch=True` flag loads model weights immediately (instead of lazy loading on first trace).

In [ ]:
from nnsight import LanguageModel
from transformers import AutoModelForSeq2SeqLM

t5 = LanguageModel(
    "google-t5/t5-large",
    automodel=AutoModelForSeq2SeqLM,
    dispatch=True,
)

t5_model = t5.model  # the underlying HuggingFace model
t5_model.eval()
t5_model.to(device)

print(f"Model loaded: google-t5/t5-large")
print(f"  d_model = {t5_model.config.d_model}")
print(f"  d_ff = {t5_model.config.d_ff}")
print(f"  num_heads = {t5_model.config.num_heads}")
print(f"  encoder_layers = {t5_model.config.num_layers}")
print(f"  decoder_layers = {t5_model.config.num_layers}")
print(f"  vocab_size = {t5_model.config.vocab_size}")

## 3. Explore Model Structure

Before training, let's understand the T5 encoder block structure and verify nnsight can capture activations at the correct hook point.

### T5 Encoder Block Architecture
Each encoder block (`T5Block`) contains:
- `layer[0]`: `T5Attention` (self-attention)
- `layer[1]`: `T5DenseActDense` (MLP: Linear(d_model->d_ff) -> ReLU/gated -> Linear(d_ff->d_model))

Our SAE targets **`layer[1]`** (the MLP output) at **block 12** (midpoint of 24 layers).

In [ ]:
# Print encoder block structure for block 0 (representative)
block0 = t5_model.encoder.block[0]
print("Encoder Block 0 structure:")
for i, layer in enumerate(block0.layer):
    print(f"  layer[{i}]: {layer.__class__.__name__}")

print(f"\nTotal encoder blocks: {len(t5_model.encoder.block)}")
print(f"Total decoder blocks: {len(t5_model.decoder.block)}")

# Show decoder block structure
dblock0 = t5_model.decoder.block[0]
print("\nDecoder Block 0 structure:")
for i, layer in enumerate(dblock0.layer):
    print(f"  layer[{i}]: {layer.__class__.__name__}")

In [ ]:
# Verify activation capture with nnsight
TARGET_BLOCK = 12
TARGET_LAYER = 1  # T5DenseActDense (MLP)

test_text = "The quick brown fox jumps over the lazy dog."

with t5.trace(test_text):
    mlp_output = t5_model.encoder.block[TARGET_BLOCK].layer[TARGET_LAYER].output.save()

print(f"Captured MLP output shape: {mlp_output.value.shape}")
print(f"Captured MLP output dtype: {mlp_output.value.dtype}")
print(f"Activation statistics:")
print(f"  mean: {mlp_output.value.float().mean():.4f}")
print(f"  std: {mlp_output.value.float().std():.4f}")
print(f"  L2 norm (per token): {mlp_output.value.float().norm(dim=-1).mean():.4f}")
print(f"  sqrt(d_model) = {1024**0.5:.4f}")

## 4. Dataset and Activation Collection

We use the C4 (Colossal Clean Crawled Corpus) dataset from HuggingFace, which is a standard pre-training corpus. We stream the dataset to avoid downloading the full 300GB+.

The activation collection pipeline:
1. Sample text from C4
2. Tokenize with T5's tokenizer
3. Run through T5 encoder with nnsight
4. Capture MLP activations at the target layer
5. Flatten to [n_tokens, d_model] for SAE training

In [ ]:
from datasets import load_dataset

# Load C4 in streaming mode
dataset = load_dataset("allenai/c4", "en", split="train", streaming=True)

# Get a sample to verify
sample = next(iter(dataset))
print(f"Sample text length: {len(sample['text'])} chars")
print(f"Sample preview: {sample['text'][:200]}...")

# T5 tokenizer
tokenizer = t5.tokenizer
print(f"\nTokenizer vocab size: {tokenizer.vocab_size}")
print(f"Tokenizer pad_token_id: {tokenizer.pad_token_id}")
print(f"Tokenizer eos_token_id: {tokenizer.eos_token_id}")

## 5. Hyperparameters

Key choices following SAELens conventions:
- **d_sae = 16384**: 16x expansion over d_model=1024, providing rich feature space
- **L1 coefficient = 5**: Controls sparsity, matching the SAELens tutorial
- **Activation normalization**: `expected_average_only_in` scales activations so expected L2 norm = sqrt(d_in)
- **Decoder init norm = 0.1**: Rows of W_dec normalized to 0.1, then W_enc = W_dec.T
- **L1 warm-up**: 5% of training to avoid dead features early on

In [ ]:
@dataclass
class SAEConfig:
    # Model dimensions
    d_in: int = 1024       # T5-large d_model
    d_sae: int = 16384     # SAE bottleneck width

    # Training
    lr: float = 1e-4
    batch_size: int = 4096         # total tokens per batch
    context_size: int = 128        # tokens per prompt
    total_steps: int = 50_000

    # SAE-specific
    l1_coefficient: float = 5.0
    l1_warm_up_steps: int = 2_500  # 5% of total_steps
    decoder_init_norm: float = 0.1
    apply_b_dec_to_input: bool = False

    # Activation normalization
    normalize_activations: bool = True
    n_batches_for_norm_estimate: int = 100

    # Target layer
    target_block: int = 12
    target_layer: int = 1  # T5DenseActDense MLP

    # Logging
    log_every: int = 100

    # Device
    device: str = device
    dtype: str = "float32"

config = SAEConfig()
print(f"SAE Config:")
print(f"  d_in={config.d_in}, d_sae={config.d_sae}")
print(f"  Expansion ratio: {config.d_sae / config.d_in:.1f}x")
print(f"  Total parameters: {config.d_in * config.d_sae * 2 + config.d_sae + config.d_in:,}")
print(f"  L1 coefficient: {config.l1_coefficient}")
print(f"  L1 warm-up steps: {config.l1_warm_up_steps}")

## 6. Sparse Autoencoder Implementation

This implementation follows SAELens conventions exactly:

**Architecture:**
```
x -> [optional bias subtraction] -> normalize -> encode -> relu -> decode -> output
```

**Initialization:**
1. W_dec: kaiming_uniform, then normalize rows to `decoder_init_norm` (0.1)
2. W_enc = W_dec.T
3. b_enc: zeros
4. b_dec: zeros

**Loss:**
```
total_loss = mse_loss + l1_coefficient * weighted_l1_loss
mse_loss = (reconstruction - input)^2 .sum(-1).mean()
l1_loss = (feature_acts * W_dec.norm(dim=1)).norm(1, dim=-1).mean()
```

In [ ]:
class SparseAutoencoder(nn.Module):
    """Sparse Autoencoder following SAELens conventions."""

    def __init__(self, cfg: SAEConfig):
        super().__init__()
        self.cfg = cfg
        self.dtype = getattr(torch, cfg.dtype)

        # Parameters
        self.W_dec = nn.Parameter(torch.empty(cfg.d_sae, cfg.d_in, dtype=self.dtype))
        self.W_enc = nn.Parameter(torch.empty(cfg.d_in, cfg.d_sae, dtype=self.dtype))
        self.b_enc = nn.Parameter(torch.zeros(cfg.d_sae, dtype=self.dtype))
        self.b_dec = nn.Parameter(torch.zeros(cfg.d_in, dtype=self.dtype))

        # Activation normalization factor
        self.register_buffer("scaling_factor", torch.tensor(1.0))

        # Initialize weights
        self._initialize_weights()

    def _initialize_weights(self):
        """Initialize following SAELens conventions."""
        # W_dec: kaiming uniform, then normalize rows to 0.1
        nn.init.kaiming_uniform_(self.W_dec.data)
        with torch.no_grad():
            self.W_dec.data /= self.W_dec.norm(dim=-1, keepdim=True)
            self.W_dec.data *= self.cfg.decoder_init_norm

        # W_enc = W_dec.T
        self.W_enc.data = self.W_dec.data.T.clone().detach().contiguous()

    def estimate_scaling_factor(self, data_loader, n_batches=100):
        """Estimate activation scaling factor for expected_average_only_in normalization."""
        if not self.cfg.normalize_activations:
            return

        norms = []
        for i, batch in enumerate(data_loader):
            if i >= n_batches:
                break
            norms.append(batch.float().norm(dim=-1).mean().item())

        mean_norm = np.mean(norms)
        self.scaling_factor = torch.tensor(
            (self.cfg.d_in ** 0.5) / mean_norm,
            device=self.cfg.device
        )
        print(f"Estimated scaling factor: {self.scaling_factor.item():.4f}")
        print(f"  Mean activation L2 norm: {mean_norm:.4f}")
        print(f"  Target norm (sqrt(d_in)): {self.cfg.d_in ** 0.5:.4f}")

    def encode(self, x: torch.Tensor):
        """Encode input to feature activations. Returns: (feature_acts, hidden_pre)"""
        if self.cfg.normalize_activations and self.scaling_factor > 0:
            x = x * self.scaling_factor

        if self.cfg.apply_b_dec_to_input:
            x = x - self.b_dec

        hidden_pre = x @ self.W_enc + self.b_enc
        feature_acts = F.relu(hidden_pre)
        return feature_acts, hidden_pre

    def decode(self, feature_acts: torch.Tensor):
        """Decode feature activations back to input space."""
        return feature_acts @ self.W_dec + self.b_dec

    def forward(self, x: torch.Tensor):
        """Forward pass. Returns: (reconstruction, feature_acts, hidden_pre)"""
        feature_acts, hidden_pre = self.encode(x)
        reconstruction = self.decode(feature_acts)
        return reconstruction, feature_acts, hidden_pre

    def calculate_loss(
        self,
        x: torch.Tensor,
        reconstruction: torch.Tensor,
        feature_acts: torch.Tensor,
        l1_coefficient: float,
    ) -> dict:
        """Calculate total loss following SAELens conventions."""
        # MSE loss
        per_item_mse = F.mse_loss(reconstruction, x, reduction="none")
        mse_loss = per_item_mse.sum(dim=-1).mean()

        # L1 loss weighted by decoder weight norms
        weighted_feature_acts = feature_acts * self.W_dec.norm(dim=1)
        l1_loss = l1_coefficient * weighted_feature_acts.norm(p=1, dim=-1).mean()

        total_loss = mse_loss + l1_loss

        # Metrics
        with torch.no_grad():
            per_token_l2_loss = per_item_mse.sum(dim=-1)
            total_variance = (x - x.mean(0)).pow(2).sum(-1)
            explained_variance = 1 - per_token_l2_loss.mean() / (total_variance.mean() + 1e-8)
            l0 = feature_acts.bool().float().sum(-1).mean()

        return {
            "total_loss": total_loss,
            "mse_loss": mse_loss,
            "l1_loss": l1_loss,
            "explained_variance": explained_variance,
            "l0": l0,
        }

    def save_model(self, path: str):
        """Save SAE weights and config."""
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)

        state_dict = {
            "W_enc": self.W_enc.data,
            "W_dec": self.W_dec.data,
            "b_enc": self.b_enc.data,
            "b_dec": self.b_dec.data,
            "scaling_factor": self.scaling_factor,
        }
        save_file(state_dict, str(path / "sae_weights.safetensors"))

        with open(path / "sae_config.json", "w") as f:
            json.dump({
                "d_in": self.cfg.d_in,
                "d_sae": self.cfg.d_sae,
                "target_block": self.cfg.target_block,
                "target_layer": self.cfg.target_layer,
                "decoder_init_norm": self.cfg.decoder_init_norm,
                "normalize_activations": self.cfg.normalize_activations,
            }, f, indent=2)

        print(f"Model saved to {path}")

    @classmethod
    def load_model(cls, path: str, cfg: SAEConfig):
        """Load SAE from disk."""
        path = Path(path)
        state_dict = load_file(str(path / "sae_weights.safetensors"))

        sae = cls(cfg)
        sae.W_enc.data = state_dict["W_enc"]
        sae.W_dec.data = state_dict["W_dec"]
        sae.b_enc.data = state_dict["b_enc"]
        sae.b_dec.data = state_dict["b_dec"]
        sae.scaling_factor = state_dict["scaling_factor"]

        print(f"Model loaded from {path}")
        return sae

## 7. Activation Collection Pipeline

The activation collection uses nnsight's trace API to capture intermediate activations during a forward pass. For each batch of text:

1. Tokenize text with T5's tokenizer
2. Run `model.trace()` with the input
3. Save `model.encoder.block[12].layer[1].output` (MLP output)
4. Retrieve the saved tensor and flatten to [n_tokens, d_model]

We collect activations into a buffer, then shuffle and batch them for SAE training.

In [ ]:
class ActivationCollector:
    """Collects activations from T5 encoder using nnsight."""

    def __init__(self, model: LanguageModel, tokenizer, config: SAEConfig):
        self.model = model
        self.tokenizer = tokenizer
        self.config = config
        self.target_block = config.target_block
        self.target_layer = config.target_layer

    @torch.no_grad()
    def collect_activations(self, texts: list[str]) -> torch.Tensor:
        """
        Collect MLP activations from T5 encoder for a batch of texts.
        Returns: Tensor of shape [n_valid_tokens, d_model]
        """
        tokens = self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.config.context_size,
        ).to(self.config.device)

        with self.model.trace(tokens.input_ids):
            mlp_output = self.model.encoder.block[
                self.target_block
            ].layer[self.target_layer].output.save()

        activations = mlp_output.value.to(dtype=torch.float32)
        mask = tokens.attention_mask.bool()
        activations_flat = activations[mask]

        return activations_flat

    def collect_batch(self, dataset_iter, target_tokens: int = 4096) -> torch.Tensor:
        """
        Collect enough activations to reach target_tokens.
        Returns: Tensor of shape [target_tokens, d_model]
        """
        collected = []
        total_tokens = 0

        while total_tokens < target_tokens:
            try:
                sample = next(dataset_iter)
                texts = [sample["text"]]
            except StopIteration:
                break

            acts = self.collect_activations(texts)
            collected.append(acts)
            total_tokens += acts.shape[0]

        if not collected:
            return torch.empty(0, self.config.d_in, device=self.config.device)

        all_acts = torch.cat(collected, dim=0)
        if all_acts.shape[0] > target_tokens:
            indices = torch.randperm(all_acts.shape[0])[:target_tokens]
            all_acts = all_acts[indices]

        return all_acts

## 8. Training Loop

The training loop follows SAELens conventions:
1. Collect activations in batches
2. Estimate activation scaling factor (for `expected_average_only_in` normalization)
3. Train SAE with MSE + L1 loss
4. Log metrics including explained variance, L0, and loss components
5. Warm up L1 coefficient over first 5% of training

In [ ]:
def train_sae(
    sae: SparseAutoencoder,
    collector: ActivationCollector,
    dataset_iter,
    config: SAEConfig,
    use_wandb: bool = False,
):
    """Train the SAE following SAELens conventions."""
    sae.to(config.device)
    sae.train()

    optimizer = torch.optim.Adam(
        sae.parameters(),
        lr=config.lr,
        betas=(0.9, 0.999),
    )

    # Estimate activation scaling factor
    if config.normalize_activations:
        print("Estimating activation scaling factor...")

        def temp_provider():
            while True:
                acts = collector.collect_batch(dataset_iter, target_tokens=4096)
                if acts.shape[0] > 0:
                    yield acts

        sae.estimate_scaling_factor(
            temp_provider(),
            n_batches=config.n_batches_for_norm_estimate,
        )

    # Training metrics
    metrics_history = {
        "step": [],
        "total_loss": [],
        "mse_loss": [],
        "l1_loss": [],
        "explained_variance": [],
        "l0": [],
        "l1_coeff": [],
    }

    print(f"\nStarting training for {config.total_steps} steps...")
    print(f"  Target block: {config.target_block}, layer: {config.target_layer}")
    print(f"  d_in={config.d_in}, d_sae={config.d_sae}")
    print(f"  L1 coefficient: {config.l1_coefficient}")
    print(f"  L1 warm-up steps: {config.l1_warm_up_steps}")
    print()

    pbar = tqdm(range(config.total_steps), desc="Training SAE")

    for step in pbar:
        # Collect activations
        batch_acts = collector.collect_batch(
            dataset_iter,
            target_tokens=config.batch_size,
        )

        if batch_acts.shape[0] == 0:
            continue

        # Forward pass
        reconstruction, feature_acts, _ = sae(batch_acts)

        # L1 warm-up
        if step < config.l1_warm_up_steps:
            l1_coeff = config.l1_coefficient * (step / config.l1_warm_up_steps)
        else:
            l1_coeff = config.l1_coefficient

        # Calculate loss
        losses = sae.calculate_loss(
            batch_acts, reconstruction, feature_acts, l1_coeff
        )

        # Backward pass
        optimizer.zero_grad()
        losses["total_loss"].backward()
        torch.nn.utils.clip_grad_norm_(sae.parameters(), 1.0)
        optimizer.step()

        # Log metrics
        metrics_history["step"].append(step)
        metrics_history["total_loss"].append(losses["total_loss"].item())
        metrics_history["mse_loss"].append(losses["mse_loss"].item())
        metrics_history["l1_loss"].append(losses["l1_loss"].item())
        metrics_history["explained_variance"].append(losses["explained_variance"].item())
        metrics_history["l0"].append(losses["l0"].item())
        metrics_history["l1_coeff"].append(l1_coeff)

        # Update progress bar
        if step % config.log_every == 0:
            pbar.set_postfix({
                "loss": f"{losses['total_loss'].item():.4f}",
                "mse": f"{losses['mse_loss'].item():.4f}",
                "l1": f"{losses['l1_loss'].item():.4f}",
                "ev": f"{losses['explained_variance'].item():.4f}",
                "l0": f"{losses['l0'].item():.1f}",
            })

        # Log to wandb
        if use_wandb and step % config.log_every == 0:
            import wandb
            wandb.log({
                "train/total_loss": losses["total_loss"].item(),
                "train/mse_loss": losses["mse_loss"].item(),
                "train/l1_loss": losses["l1_loss"].item(),
                "train/explained_variance": losses["explained_variance"].item(),
                "train/l0": losses["l0"].item(),
                "train/l1_coefficient": l1_coeff,
            }, step=step)

    pbar.close()
    return metrics_history

## 9. Run Training

This will take approximately 2-4 hours on a GPU depending on your hardware. The training collects activations from C4 text and trains the SAE to reconstruct them sparsely.

In [ ]:
# Initialize SAE
sae = SparseAutoencoder(config)
print(f"SAE initialized with {sum(p.numel() for p in sae.parameters()):,} parameters")

# Initialize activation collector
collector = ActivationCollector(t5, tokenizer, config)

# Create dataset iterator
dataset_iter = iter(dataset)

# Optional: Initialize wandb (set to True to enable)
use_wandb = False
if use_wandb:
    import wandb
    wandb.init(project="sae_t5_training", config=vars(config))

# Train
metrics = train_sae(
    sae=sae,
    collector=collector,
    dataset_iter=dataset_iter,
    config=config,
    use_wandb=use_wandb,
)

print("\nTraining complete!")

## 10. Evaluation and Analysis

Let's evaluate the trained SAE using standard metrics:
- **Explained Variance**: How much of the original activation variance is captured
- **L0**: Average number of active features per token (sparsity)
- **Feature Density**: Distribution of feature activation frequencies
- **Dead Features**: Features that never activate

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("SAE Training Metrics", fontsize=14)

axes[0, 0].plot(metrics["step"], metrics["total_loss"])
axes[0, 0].set_title("Total Loss")
axes[0, 0].set_xlabel("Step")

axes[0, 1].plot(metrics["step"], metrics["mse_loss"])
axes[0, 1].set_title("MSE Loss")
axes[0, 1].set_xlabel("Step")

axes[0, 2].plot(metrics["step"], metrics["l1_loss"])
axes[0, 2].set_title("L1 Loss")
axes[0, 2].set_xlabel("Step")

axes[1, 0].plot(metrics["step"], metrics["explained_variance"])
axes[1, 0].set_title("Explained Variance")
axes[1, 0].set_xlabel("Step")
axes[1, 0].axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target (0.8)')
axes[1, 0].legend()

axes[1, 1].plot(metrics["step"], metrics["l0"])
axes[1, 1].set_title("L0 (Active Features)")
axes[1, 1].set_xlabel("Step")

axes[1, 2].plot(metrics["step"], metrics["l1_coeff"])
axes[1, 2].set_title("L1 Coefficient")
axes[1, 2].set_xlabel("Step")

plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def evaluate_sae(sae, collector, dataset_iter, n_batches=10):
    """Evaluate SAE on held-out activations."""
    sae.eval()

    all_feature_acts = []
    all_reconstructions = []
    all_inputs = []

    for _ in range(n_batches):
        batch_acts = collector.collect_batch(dataset_iter, target_tokens=4096)
        if batch_acts.shape[0] == 0:
            continue

        reconstruction, feature_acts, _ = sae(batch_acts)
        all_feature_acts.append(feature_acts)
        all_reconstructions.append(reconstruction)
        all_inputs.append(batch_acts)

    if not all_feature_acts:
        print("No activations collected for evaluation")
        return {}

    feature_acts = torch.cat(all_feature_acts, dim=0)
    reconstructions = torch.cat(all_reconstructions, dim=0)
    inputs = torch.cat(all_inputs, dim=0)

    per_token_mse = (reconstructions - inputs).pow(2).sum(dim=-1)
    total_variance = (inputs - inputs.mean(0)).pow(2).sum(dim=-1)
    explained_variance = 1 - per_token_mse.mean() / (total_variance.mean() + 1e-8)

    active_features = feature_acts.bool().float()
    l0 = active_features.sum(-1).mean()
    feature_density = active_features.mean(0)
    dead_features = (feature_density < 1e-6).sum().item()

    eval_metrics = {
        "explained_variance": explained_variance.item(),
        "l0": l0.item(),
        "feature_density_mean": feature_density.mean().item(),
        "feature_density_std": feature_density.std().item(),
        "dead_features": dead_features,
        "total_features": feature_acts.shape[-1],
        "dead_feature_pct": dead_features / feature_acts.shape[-1] * 100,
    }

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.hist(feature_density.cpu().numpy(), bins=50, edgecolor='black')
    plt.xlabel("Feature Activation Rate")
    plt.ylabel("Count")
    plt.title("Feature Density Distribution")
    plt.axvline(x=feature_density.mean().item(), color='r', linestyle='--',
                label=f'Mean: {feature_density.mean().item():.4f}')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.hist(l0.cpu().numpy(), bins=50, edgecolor='black')
    plt.xlabel("L0 (Active Features per Token)")
    plt.ylabel("Count")
    plt.title("L0 Distribution")
    plt.axvline(x=l0.item(), color='r', linestyle='--',
                label=f'Mean: {l0.item():.1f}')
    plt.legend()

    plt.tight_layout()
    plt.show()

    return eval_metrics

# Run evaluation
eval_metrics = evaluate_sae(sae, collector, dataset_iter)
print("\nEvaluation Metrics:")
for key, value in eval_metrics.items():
    print(f"  {key}: {value}")

In [ ]:
@torch.no_grad()
def visualize_features(sae, collector, dataset_iter, n_samples=5):
    """Visualize what individual features respond to."""
    sae.eval()

    samples = []
    for _ in range(n_samples * 10):
        try:
            sample = next(dataset_iter)
            texts = [sample["text"]]
        except StopIteration:
            break

        tokens = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=config.context_size,
        ).to(config.device)

        with t5.trace(tokens.input_ids):
            mlp_output = t5_model.encoder.block[
                config.target_block
            ].layer[config.target_layer].output.save()

        activations = mlp_output.value.to(dtype=torch.float32)
        mask = tokens.attention_mask.bool()
        activations_flat = activations[mask]

        feature_acts = F.relu(activations_flat @ sae.W_enc + sae.b_enc)

        for feat_idx in range(min(10, config.d_sae)):
            feat_act = feature_acts[:, feat_idx]
            if feat_act.max() > 0:
                top_token_idx = feat_act.argmax().item()
                token_ids = tokens.input_ids[mask]
                top_token = tokenizer.decode([token_ids[top_token_idx]])
                samples.append({
                    "feature": feat_idx,
                    "token": top_token,
                    "activation": feat_act[top_token_idx].item(),
                    "text_snippet": sample["text"][:100],
                })

    if samples:
        print("Top activating features:")
        seen = set()
        for s in sorted(samples, key=lambda x: x["activation"], reverse=True):
            if s["feature"] not in seen:
                print(f"  Feature {s['feature']:5d}: '{s['token']}' "
                      f"(act={s['activation']:.4f})")
                seen.add(s["feature"])
                if len(seen) >= 10:
                    break

visualize_features(sae, collector, dataset_iter)

In [ ]:
@torch.no_grad()
def logit_lens_analysis(sae, tokenizer, n_features=10):
    """
    Project SAE decoder weights onto the unembedding matrix
    to see what tokens each feature promotes.
    """
    # T5 uses shared embeddings for input and output
    embeddings = t5_model.decoder.embed_tokens.weight  # [vocab_size, d_model]

    # Project W_dec onto vocabulary: [d_sae, vocab_size]
    projection = sae.W_dec @ embeddings.T

    top_k = 5
    vals, inds = projection.topk(top_k, dim=1)

    random_indices = torch.randint(0, projection.shape[0], (n_features,))

    print(f"Top {top_k} tokens promoted by {n_features} random features:")
    print("-" * 80)
    for idx in random_indices:
        feature_idx = idx.item()
        tokens = [tokenizer.decode([i]) for i in inds[feature_idx]]
        probs = F.softmax(vals[feature_idx], dim=0)
        token_strs = [f"'{t}' ({p:.3f})" for t, p in zip(tokens, probs)]
        print(f"Feature {feature_idx:5d}: {', '.join(token_strs)}")

    return projection

projection = logit_lens_analysis(sae, tokenizer)

## 11. Save Trained SAE

Save the trained SAE weights and configuration for later analysis or deployment.

In [ ]:
# Save the trained SAE
save_path = Path("checkpoints/sae_t5_large_encoder_block12")
sae.save_model(str(save_path))

# Save training metrics
with open(save_path / "training_metrics.json", "w") as f:
    json.dump(metrics, f)

print(f"\nAll artifacts saved to: {save_path}")
print(f"  sae_weights.safetensors")
print(f"  sae_config.json")
print(f"  training_metrics.json")

## Summary

This notebook demonstrated training a Sparse Autoencoder on T5-large encoder activations using nnsight for activation capture. Key outcomes:

1. **nnsight successfully wraps T5** via `LanguageModel` with `AutoModelForSeq2SeqLM`
2. **Activation capture** at encoder block 12 MLP output works reliably
3. **SAE training** follows SAELens conventions (initialization, loss, normalization)
4. **Evaluation metrics** provide insight into feature quality and sparsity

### Next Steps
- Experiment with different target layers (early vs. late encoder)
- Try decoder-side SAEs (requires encoder-decoder input pairs)
- Compare feature quality across layers
- Investigate dead feature mitigation strategies
- Use the trained SAE for mechanistic interpretability

### References
- [SAELens](https://github.com/decoderesearch/SAELens)
- [nnsight](https://nnsight.net/)
- [Anthropic's SAE work](https://transformer-circuits.pub/2024/april-update/index.html)
- [Scaling Monosemanticity](https://www.anthropic.com/research/mapping-mind-language-model)